# <center>Procesamiento Digital de Señales de Audio</center>
### <center>Instituto de Ingeniería Eléctrica - UdelaR</center>
## <center> Análisis por Predicción Lineal </center>

### Como correr este notebook

Es posible descargarlo y correrlo localmente en su computadora

Tambien pueden correrlo en Google Colab usando el siguiente link.

<table align="center">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/emidan19/audio-dsp/blob/main/practicos/P3b_lpc.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Correr en Google Colab</a>
  </td>
</table>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from  matplotlib import patches
from matplotlib.pyplot import axvline, axhline
from scipy import io, signal
from scipy.linalg import solve_toeplitz
from scipy.io.wavfile import read
from IPython.display import Audio
import pandas as pd

## Parte 1

En el modelo de predicción lineal se asume que la muestra actual de la señal de voz $s[n]$ es predecible a partir de una combinación lineal de $p$ muestras previas,

$$\tilde{s}[n] = \sum_{k=1}^{p}\alpha_k s[n-k]$$

El problema consiste en encontrar los coeficientes $\alpha_k$ del predictor que mejor aproximan a la señal $s[n]$. Para eso se define el error de predicción como

$$e_n[m] = s_n[m] - \tilde{s}_n[m]$$

donde $s_n[m]$ es un fragmento de tiempo corto de la señal de voz elegido en torno a la muestra $n$.

Se define el error cuadrático medio de predicción como

$$E_n = \sum_m e_n^2[m]$$

para algún intervalo de muestras $m$ que no es necesario especificar por el momento.
En el modelo de predicción lineal, el conjunto de coeficientes $\lbrace\hat{\alpha}_k\rbrace$ óptimo es el que minimiza el error cuadrático medio de predicción. Se pide:


1. Demostrar que los coeficientes que minimizan el error cuadrático medio obedecen el siguiente sistema lineal de ecuaciones (*ecuaciones normales*),

    $$\sum_{k=1}^{p}\hat{\alpha}_k\sum_m s_n[m-i]s_n[m-k]=\sum_m s_n[m-i]s_n[m],\,\,\,1\leq i \leq p$$

1. Demostrar que el error cuadrático medio mínimo de predicción es

    $$E_n = \sum_m s_n^2[m]-\sum_{k=1}^p\hat{\alpha}_k\sum_m s_n[m]s_n[m-k]$$


**Parte 1.1)**

...

**Parte 1.2)**

...


## Parte 2

En este problema se aplica la técnica de LPC para la clasificación de vocales, usando una base de datos de vocales aisladas pronunciadas por dos hablantes.

El procedimiento consiste en calcular el modelo todo polos de la señal de voz, y a partir de los polos obtener la frecuencia de las dos primeras formantes $\left(F_1,\,F_2\right)$.


A modo de referencia, en el cuadro de abajo se indica la frecuencia promedio de las dos primeras formantes de las vocales del idioma español (Estos datos son aproximaciones de los datos provistos en https://joaquimllisterri.cat/phonetics/fon_anal_acus/caract_acust.html).


La señal analizada puede clasificarse a partir la vocal de referencia mas cercana en el plano $\left(F_1,\,F_2\right)$.

| Fonema | $F_1(Hz)$ | $F_2(Hz)$ |
|:------:|:---------:|:---------:|
|   /a/  |    800    |    1170   |
|   /e/  |    480    |    2300   |
|   /i/  |    240    |    2800   |
|   /o/  |    510    |    960    |
|   /u/  |    250    |    630    |

__Primeras dos formantes de las vocales en el idioma español.__


1. Implementar un algoritmo para procesar todas las señales de la base de datos, calculando para cada una la frecuencia de las dos primeras formantes. Mostrar los resultados como un mapa de formantes en el plano $\left(F_1,\,F_2\right)$.
1. Clasificar las señales a partir de las vocales de referencia. Reportar la tasa de acierto obtenida para cada vocal y para cada hablante.
1. Analizar los resultados y proponer alguna estrategia para mejorarlos.
    

Las señales de la base de datos están muestreadas a 8000 Hz. Hay un directorio por hablante y el nombre de los archivo de audio es **[vocal]-[número].wav**, con **número** de 1 a 10.
Los archivos contienen un único fonema, de duración variable (sin silencio al comienzo o al final), pero todos superan las 550 muestras.


Tener en cuenta los siguientes aspectos.

- Se sugiere tomar una ventana centrada en la muestra central de cada señal.
- Elegir adecuadamente el tamaño de la ventana $N$ y el orden $p$ del modelo. Para éste último puede ser útil la elección según el error normalizado $E_p$.
- Calcular los polos y representarlos en un diagrama de polos y ceros.
- Eliminar los polos reales y los polos con $\omega\geq\pi$ (Son redundantes por ser complejos conjugados de los polos con $\omega<\pi$). Eliminar los polos de ancho de banda mayor a cierto umbral. Elegir adecuadamente el umbral.
- Establecer la frecuencia de las dos primeras formantes como la frecuencia de los dos polos de menor frecuencia.


Descargar los audios de vocales

In [ ]:
# !pip install wget # Descomentar en caso de usar Google Colab
import wget
from pathlib import Path

In [ ]:
vocales = ['a','e','i','o','u']
hablantes = ['martin','cecilia']
indices = range(1,11)

url = "https://raw.githubusercontent.com/emidan19/audio-dsp/main/practicos/Archivos_P3/problema2/vocales/"

for hablante in hablantes:
  carpeta = Path('./problema2/vocales') / hablante
  carpeta.mkdir(parents=True, exist_ok=True)
  for vocal in vocales:
    for indice in indices:
      archivo = f"{vocal}-{indice}.wav"
      destino = carpeta / archivo
      # Descargar solo si no existe
      if not destino.exists():
        wget.download(url + hablante + '/' + archivo + "?raw=true", out=str(destino))
        print(f"\nDescarga completada: {hablante}/{archivo}")
      else:
        print(f"El archivo ya existe: {hablante}/{archivo}")

### Cargar audios

In [ ]:
### Cargar los audios y asignarle una etiqueta del tipo #hablante-vocal
folders = ['./problema2/vocales/martin/','./problema2/vocales/cecilia/']

files_list = []
labels = []

for folder in folders:
  for vocal in vocales:
    for indice in indices:
      _, audio = io.wavfile.read(folder+vocal+'-'+str(indice)+'.wav')
      files_list.append(audio)
      labels.append(str(folders.index(folder))+'-'+vocal)
# Frecuencia de muestreo
fs = 8000

In [ ]:
# Esuchar vocal del hablante 1
Audio(files_list[45], rate=fs)

In [ ]:
# Esuchar vocal del hablante 2
Audio(files_list[95], rate=fs)

### Definir funciones para la extracción de formantes

In [ ]:
# Reutilización de código usado en clase (LPC_analysis.ipynb)

def lpc_analysis(s, p=20):
    """ compute the LPC analysis using the autocorrelation method

    Parameters
    ----------
    s : numpy array
        windowed signal frame as a numpy 1D array.
    p : int
        model order.

    Returns
    -------
    ak : numpy array
         model coefficients.
    e : float
        minimum mean squared error.
    e_norm : float
             normalized minimum mean squared error.
    """
    # compute autocorrelation values
    r = np.zeros(p+1)

    r[0] = np.sum(s**2)
    for k in (np.arange(p)+1):
      r[k] = ... # COMPLETAR

    # solve to compute model coefficients
    ak = solve_toeplitz( ... ) # COMPLETAR

    # compute mean squared error
    e = ... # COMPLETAR

    # compute normalized mean squared error
    e_norm = e / r[0]

    return ak, e, e_norm

#### 0. Elegir un audio ejemplo

In [ ]:
ind_ejemplo = 54
x = files_list[ind_ejemplo]
print('Hablante-vocal:', labels[ind_ejemplo])
Audio(x, rate=fs)

#### 1. Ventana centrada en la muestra central de la señal

In [ ]:
def frame_central(x, N):
  """ Extrae un frame de N muestras centrado en la muestra central de x """
  centro = len(x) // 2
  inicio = centro - N // 2
  return x[inicio:inicio + N]


def enventanar(s, ventana='hann'):
  """ Aplica una ventana suavizante al frame s. Devuelve el frame enventanado y la ventana """
  window = signal.windows.get_window(ventana, len(s))
  return s * window, window

In [ ]:
N = 550
s = frame_central(x, N=N)
s_win, window = enventanar(s)

plt.figure(figsize=(12, 6))

# mostrar frame central de la señal
plt.subplot(2, 1, 1)
plt.plot(s, 'k')
plt.title('Frame central de la señal')
plt.ylabel('Amplitud')
plt.xlabel('Tiempo (muestras)')

# mostrar frame enventanado y venana
plt.subplot(2, 1, 2)
plt.plot(s_win / np.max(np.abs(s_win)), 'k', label='Frame enventanado (normalizado)')
plt.plot(window, 'r', label='Ventana suavizante')
plt.title('Frame central de la señal enventanado')
plt.ylabel('Amplitud')
plt.xlabel('Tiempo (muestras)')
plt.legend()

plt.tight_layout()
plt.show()

#### 2. Orden $p$ del modelo según el error normalizado $E_p$

In [ ]:
def error_normalizado_vs_orden(s_win, p_max):
  """ Error normalizado de predicción E_p para los órdenes p = 1, ..., p_max """
  # Calcular el analisis LPC para todos numeros de polos

  # Devolver error normalizado en un array

  return ...

In [ ]:
# Graficar error normalizado
p_max = 20
ordenes = np.arange(1, p_max + 1)
Ep = error_normalizado_vs_orden(s_win, p_max)

plt.figure(figsize=(10, 4))
plt.plot(ordenes, Ep, 'k-o')
plt.title('Error normalizado de predicción $E_p$')
plt.xlabel('Orden p')
plt.ylabel('$E_p$')
plt.xticks(ordenes)
plt.grid(alpha=0.3)

In [ ]:
# Encontrar el número óptimo de polos
# encontrando el codo de la curva con una tolerancia delta

def encontrar_orden_optimo(s_win, p_max, delta, p_min=8):

    # Curva de error: Ep[k] corresponde al orden k+1
    Ep = error_normalizado_vs_orden(s_win, p_max)
    #Inicializar con p_max
    p_opt = p_max

    # Hallar codo: Ep[p] / Ep[p-1] = E_{p+1} / E_p
    for p in range(p_min, p_max):
        if ... : # COMPLETAR CONDICION
            p_opt = p
            break

    return p_opt

In [ ]:
delta = 0.15
p_opt = encontrar_orden_optimo(s_win, p_max, delta)
print(f'Orden óptimo para el audio ejemplo: p = {p_opt}')

plt.figure(figsize=(10, 4))
plt.plot(ordenes, Ep, 'k-o')
plt.axvline(p_opt, color='b', ls='--', label=f'p = {p_opt} (codo con tolerancia $\\delta$ = {delta})')
plt.title('Error normalizado de predicción $E_p$ del audio ejemplo')
plt.xlabel('Orden p')
plt.ylabel('$E_p$')
plt.xticks(ordenes)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# En lo que sigue se usa el orden encontrado automáticamente,
# que es el que usa lpc_f1_f2_vocal_detect para cada audio
p = p_opt

#### 3. Polos del modelo y diagrama de polos y ceros

In [ ]:
def polos_lpc(ak):
  """ Polos del modelo todo polos H(z) = G / A(z), con A(z) = 1 - sum_k a_k z^{-k} """
  # Devolver los polos de H(z) (raíces de A(z) )

  roots_Az = ... # COMPLETAR

  return roots_Az

In [ ]:
def plot_polos_ceros(polos, p, polos_formantes=None):
  """ Diagrama de polos y ceros del modelo LPC. Si se pasan polos_formantes,
  se resaltan los polos que se conservan como candidatos a formante """
  plt.figure(figsize=(6, 6))
  ax = plt.gca()
  # circunferencia unidad y ejes
  unit_circle = patches.Circle((0,0), radius=1, fill=False,
                               color='black', ls='solid', alpha=0.6)
  ax.add_patch(unit_circle)
  axvline(0, color='0.6')
  axhline(0, color='0.6')

  # H(z) = G z^p / (z^p - a_1 z^{p-1} - ... - a_p): p ceros en el origen
  plt.plot(0, 0, 'ko', ms=12, mfc='none', mew=2, label=f'Ceros (orden {p}) en el origen')
  if polos_formantes is None:
    plt.plot(polos.real, polos.imag, 'rx', ms=12, mew=3, label='Polos')
  else:
    polos_descartados = polos[~np.isin(polos, polos_formantes)]
    plt.plot(polos_descartados.real, polos_descartados.imag, 'x', color='0.5',
             ms=12, mew=3, label='Polos descartados')
    plt.plot(polos_formantes.real, polos_formantes.imag, 'rx',
             ms=12, mew=3, label='Polos candidatos a formante')

  ax.set_aspect('equal')
  plt.xlim(-1.2, 1.2)
  plt.ylim(-1.2, 1.2)
  plt.title("Polos en plano Z")
  plt.xlabel("Real")
  plt.ylabel("Imaginario")
  plt.legend(loc='lower left', fontsize=9)
  plt.show()

In [ ]:
# Análisis LPC del audio ejemplo con el orden elegido y cálculo de los polos
ak, e, e_norm = lpc_analysis(s_win, p)
polos = polos_lpc(ak)

plot_polos_ceros(polos, p)

In [ ]:
# Graficar espectro de la ventana y transferencia H(z)
Ndft = 1024  # mayor o igual al largo de la ventana N

# filtro todo polos: H(z) = G / A(z)
G = np.sqrt(e)
a = np.concatenate([[1], -ak])

X = np.fft.fft(s_win, Ndft)
w, H = signal.freqz(G, a, worN=Ndft, whole=True)
f = w / (2 * np.pi) * fs
ind_fmx = Ndft // 2

plt.figure(figsize=(10, 4))
plt.plot(f[:ind_fmx], 20 * np.log10(np.abs(X[:ind_fmx])), 'k', label='Espectro señal enventanada')
plt.plot(f[:ind_fmx], 20 * np.log10(np.abs(H[:ind_fmx])), 'r', label=f'Respuesta en frecuencia LPC (p = {p})')
# frecuencia de los polos con 0 < ω < π
for fk in np.angle(polos[np.imag(polos) > 0]) * fs / (2 * np.pi):
  plt.axvline(fk, color='0.5', ls=':')
plt.title('Espectro de la señal y respuesta en frecuencia del filtro LPC (punteado: frecuencia de los polos)')
plt.ylabel('Magnitud (dB)')
plt.xlabel('Frecuencia (Hz)')
plt.legend(loc='lower left')
plt.show()

#### 4. Eliminación de polos que no corresponden a formantes

Se eliminan los polos reales, los polos con $\omega\geq\pi$ (complejos conjugados de los polos con $\omega<\pi$) y los polos con ancho de banda mayor a un umbral.

In [ ]:
def frecuencia_y_ancho_de_banda(polos, fs):
  """ Frecuencia (Hz) y ancho de banda (Hz) asociados a cada polo z = r e^{jθ} """
  frecs = ... # COMPLETAR
  bws = ... # COMPLETAR
  return frecs, bws

def filtrar_polos(polos, fs, bw_max=400, f_min=200):
  """ Se queda con los polos candidatos a formante """
  frecs, bws = frecuencia_y_ancho_de_banda(polos, fs)

  # Eliminar polos reales y polos con ω >= π (imag(z) <= 0), que son
  # redundantes por ser complejos conjugados de los polos con ω < π
  complejos = ... # COMPLETAR CONDICIÓN

  # Eliminar polos de ancho de banda mayor al umbral (resonancias poco marcadas)
  angostos = ... # COMPLETAR CONDICIÓN

  # Eliminar polos cercanos a continua (inclinación espectral del pulso glótico)
  no_dc = ... # COMPLETAR CONDICIÓN

  return polos[complejos & angostos & no_dc]

In [ ]:
# Umbrales para el filtrado de polos
bw_max = 400  # Hz
f_min = 200   # Hz

polos_formantes = filtrar_polos(polos, fs, bw_max, f_min)

plot_polos_ceros(polos, p, polos_formantes)

# Frecuencia y ancho de banda de cada polo, y si se conserva como candidato a formante
frecs, bws = frecuencia_y_ancho_de_banda(polos, fs)
tabla_polos = pd.DataFrame({'Frecuencia (Hz)': frecs,
                            'Ancho de banda (Hz)': bws,
                            'Candidato a formante': np.isin(polos, polos_formantes)})
tabla_polos.sort_values('Frecuencia (Hz)').round(1)

#### 5. Frecuencia de las dos primeras formantes

In [ ]:
def primeras_formantes(polos, fs, n=2):
  """ Frecuencia (Hz) de los n polos de menor frecuencia: F1, ..., Fn.
  Si quedan menos de n polos se completa con NaN """
  frecs, _ = frecuencia_y_ancho_de_banda(polos, fs)
  frecs = np.sort(frecs)[:n]
  return np.pad(frecs, (0, n - len(frecs)), constant_values=np.nan)

In [ ]:
f1, f2 = primeras_formantes(polos_formantes, fs, n=2)

print('Hablante-vocal:', labels[ind_ejemplo])
print(f'F1 = {f1:.0f} Hz')
print(f'F2 = {f2:.0f} Hz')

#### Gráficas del análisis LPC

In [ ]:
def plot_LPC_analysis(s, s_win, window, ak, e, fs, formantes=(), Ndft=1024):
  # filtro todo polos obtenido del análisis LPC: H(z) = G / A(z)
  G = np.sqrt(e)
  a = np.concatenate([[1], -ak])

  # espectro del frame enventanado y respuesta en frecuencia del filtro
  X = np.fft.fft(s_win, Ndft)
  w, H = signal.freqz(G, a, worN=Ndft, whole=True)
  f = w / (2 * np.pi) * fs
  ind_fmx = Ndft // 2

  # respuesta al impulso del filtro
  delta = np.zeros(len(s))
  delta[0] = 1
  h = signal.lfilter([G], a, delta)

  fig, axs = plt.subplots(2, 2, figsize=(14, 7))

  # frame central de la señal
  axs[0,0].plot(s, 'k')
  axs[0,0].set_title('Frame central de la señal')
  axs[0,0].set_ylabel('Amplitud')
  axs[0,0].set_xlabel('Tiempo (muestras)')

  # frame enventanado
  axs[1,0].plot(s_win / np.max(np.abs(s_win)), 'k', label='Frame enventanado (normalizado)')
  axs[1,0].plot(window, 'r', label='Ventana suavizante')
  axs[1,0].set_title('Frame central de la señal enventanado')
  axs[1,0].set_ylabel('Amplitud')
  axs[1,0].set_xlabel('Tiempo (muestras)')
  axs[1,0].legend()

  # espectro y respuesta en frecuencia del filtro LPC
  axs[0,1].plot(f[:ind_fmx], 20 * np.log10(np.abs(X[:ind_fmx])), 'k', label='Espectro señal enventanada')
  axs[0,1].plot(f[:ind_fmx], 20 * np.log10(np.abs(H[:ind_fmx])), 'r', label='Respuesta en frecuencia LPC')
  for i, Fi in enumerate(formantes):
    axs[0,1].axvline(Fi, color='b', ls='--', lw=1)
    axs[0,1].annotate(f'F{i+1}', (Fi, 1), xycoords=('data', 'axes fraction'),
                      xytext=(3, -12), textcoords='offset points', color='b')
  axs[0,1].set_title('Espectro de la señal y respuesta en frecuencia del filtro LPC')
  axs[0,1].set_ylabel('Magnitud (dB)')
  axs[0,1].set_xlabel('Frecuencia (Hz)')
  axs[0,1].legend(loc='lower left')

  # respuesta al impulso del filtro LPC
  axs[1,1].plot(h, 'k')
  axs[1,1].set_title('Respuesta al impulso del filtro LPC')
  axs[1,1].set_ylabel('Amplitud')
  axs[1,1].set_xlabel('Tiempo (muestras)')

  fig.tight_layout()
  plt.show()

#### Detección de $F_1$ y $F_2$

El siguiente código incorpora el procedimiento para estimar las primeras dos formantes para un segmento de audio

In [ ]:
def lpc_f1_f2_vocal_detect(x, fs, p=None, p_max=20, delta=0.15, N=550, ventana='hann',
                           bw_max=400, f_min=200, plot_stuff=False):
  """ Estima las dos primeras formantes de una vocal a partir del modelo LPC

  Parameters
  ----------
  x : numpy array
      señal de la vocal.
  fs : int
      frecuencia de muestreo (Hz).
  p : int or None
      orden del modelo LPC. Si es None, se elige el orden óptimo para esta
      señal con encontrar_orden_optimo.
  p_max : int
      orden máximo considerado al elegir p.
  delta : float
      tolerancia relativa para el codo de la curva E_p: 1 - E_{p+1}/E_p < delta.
  N : int
      largo de la ventana (muestras).
  ventana : str
      tipo de ventana suavizante.
  bw_max : float
      umbral de ancho de banda de los polos (Hz).
  f_min : float
      frecuencia mínima admitida para una formante (Hz).
  plot_stuff : bool
      si es True se grafica el análisis y el diagrama de polos y ceros.

  Returns
  -------
  f1, f2 : float
      frecuencia de las dos primeras formantes (Hz).
  """
  # 1. Frame de N muestras centrado en la muestra central de la señal, enventanado
  s = frame_central(x, N)
  s_win, window = enventanar(s, ventana)

  # 2. Orden del modelo: el óptimo para esta señal según E_p (si no se fija p)
  if p is None:
    p = encontrar_orden_optimo(s_win, p_max, delta)

  # Análisis LPC de orden p
  ak, e, e_norm = lpc_analysis(s_win, p)

  # 3. Polos del modelo todo polos
  polos = polos_lpc(ak)

  # 4. Eliminar polos reales, con ω >= π y de ancho de banda mayor al umbral
  polos_formantes = filtrar_polos(polos, fs, bw_max, f_min)

  # 5. f1 y f2: frecuencia de los dos polos de menor frecuencia
  f1, f2 = primeras_formantes(polos_formantes, fs, n=2)

  if plot_stuff:
    print(f'Orden del modelo: p = {p}')
    plot_LPC_analysis(s, s_win, window, ak, e, fs, formantes=(f1, f2))
    plot_polos_ceros(polos, p, polos_formantes)

  return f1, f2

### Elección de parámetros

In [ ]:
# Parámetros del análisis
N = 550           # largo de la ventana
ventana = 'hann'  # ventana suavizante
p_max = 20        # orden máximo considerado al elegir el orden p de cada señal
delta = 0.15      # tolerancia relativa para el codo de la curva E_p: 1 - E_{p+1}/E_p < delta
bw_max = 400      # umbral de ancho de banda de los polos (Hz)
f_min = 200       # frecuencia mínima admitida para una formante (Hz)

In [ ]:
# Error normalizado E_p en función del orden p y orden óptimo elegido, para el frame central de cada señal
ordenes = np.arange(1, p_max + 1)
s_wins = [enventanar(frame_central(audio, N), ventana)[0] for audio in files_list]
Ep_db = np.array([10 * np.log10(error_normalizado_vs_orden(sw, p_max)) for sw in s_wins])
p_opts = np.array([encontrar_orden_optimo(sw, p_max, delta) for sw in s_wins])

fig, axs = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={'width_ratios': [2, 1]})

axs[0].plot(ordenes, Ep_db.T, color='0.8', lw=0.8)
axs[0].plot(ordenes, np.mean(Ep_db, axis=0), 'k', lw=2, label='Promedio (dB)')
axs[0].plot(p_opts, Ep_db[np.arange(len(p_opts)), p_opts - 1], 'r.', label='Orden óptimo de cada señal')
axs[0].set_title('Error normalizado de predicción $E_p$ (frame central de cada señal)')
axs[0].set_xlabel('Orden p')
axs[0].set_ylabel('$E_p$ (dB)')
axs[0].set_xticks(ordenes)
axs[0].legend()

axs[1].hist(p_opts, bins=np.arange(0.5, p_max + 1.5), color='0.6', edgecolor='k')
axs[1].set_title('Orden óptimo elegido para cada señal')
axs[1].set_xlabel('Orden p')
axs[1].set_ylabel('Cantidad de señales')
axs[1].set_xticks(ordenes)

fig.tight_layout()
plt.show()

In [ ]:
# Análisis de ejemplo (audio elegido en 0.)
f1, f2 = lpc_f1_f2_vocal_detect(x, fs, p_max=p_max, delta=delta, N=N, ventana=ventana,
                                bw_max=bw_max, f_min=f_min, plot_stuff=True)
print(f1)
print(f2)

### Cálculo de formantes para todos los audios

In [ ]:
f1_f2_map = []
# Crear el mapa de formantes (el orden p se elige para cada audio)
for audio in files_list:
  f1_f2_map.append(lpc_f1_f2_vocal_detect(audio, fs, p_max=p_max, delta=delta, N=N, ventana=ventana,
                                          bw_max=bw_max, f_min=f_min))

In [ ]:
Fs_a = (800,1170)
Fs_e = (480,2300)
Fs_i = (240,2800)
Fs_o = (510,960)
Fs_u = (250,630)

Fs_ref = [Fs_a,Fs_e,Fs_i,Fs_o,Fs_u]

fsize = 20

plt.figure(figsize=(10,8))
plt.title('Mapa de formantes (Notacion: x=Hablante 1 , o=Hablante 2)')
plt.plot([Fs_a[0],Fs_a[0]],[Fs_a[1],Fs_a[1]],'*r',label='Fonema de referencia \'A\' ')
plt.annotate("A", (800, 1170), fontsize=fsize)
plt.plot([Fs_e[0],Fs_e[0]],[Fs_e[1],Fs_e[1]],'*k',label='Fonema de referencia \'E\' ')
plt.annotate("E", (480, 2300), fontsize=fsize)
plt.plot([Fs_i[0],Fs_i[0]],[Fs_i[1],Fs_i[1]],'*b',label='Fonema de referencia \'I\' ')
plt.annotate("I", (240, 2800), fontsize=fsize)
plt.plot([Fs_o[0],Fs_o[0]],[Fs_o[1],Fs_o[1]],'*m',label='Fonema de referencia \'O\' ')
plt.annotate("O", (510, 960), fontsize=fsize)
plt.plot([Fs_u[0],Fs_u[0]],[Fs_u[1],Fs_u[1]],'*g',label='Fonema de referencia \'U\' ')
plt.annotate("U", (250, 630), fontsize=fsize)

for ind in np.arange(len(f1_f2_map)):
  color = 'r'*(labels[ind][-1]=='a') + 'k'*(labels[ind][-1]=='e') + 'b'*(labels[ind][-1]=='i') + 'm'*(labels[ind][-1]=='o') + 'g'*(labels[ind][-1]=='u')
  hablante = 'x'*(labels[ind][0]=='0') + 'o'*(labels[ind][0]=='1')
  f1 , f2 = f1_f2_map[ind]
  plt.plot([f1,f1], [f2,f2], hablante+color)

plt.xlabel('f1')
plt.ylabel('f2')
plt.legend()
plt.show()

### Acierto en predicción de vocales

In [ ]:
label_predict = []

for i, (f1_pred, f2_pred) in enumerate(f1_f2_map):
  dists = []
  # Calcular distancias de formantes a las de referencia
  # correspondiente a cada vocal
  for f1_ref, f2_ref in Fs_ref:
    #Usar distancia euclidea tradicional
    dist = ... # COMPLETAR
    dists.append( dist )

  # Quedarse con el indice de la distancia menor
  k_min = np.argmin(dists)

  # Asignar etiqueta de la vocal
  if i < 50:
    label_predict.append('0-'+vocales[k_min])
  else:
    label_predict.append('1-'+vocales[k_min])


In [ ]:
labels = np.array(labels)
labels_pred = np.array(label_predict)
accuracy = np.count_nonzero(np.array(labels)==np.array(label_predict))
print('Tasa de aciertos: '+str(accuracy)+'%')

for i in (np.arange(10)):
  # print de los aciertos en cada vocal
  if i==0:
    print('\nAciertos hablante1:')
  if i==5:
    print('\nAciertos hablante2:')
  print(f'Vocal "{vocales[np.mod(i,5)]}":',np.count_nonzero(labels[10*i:10*i+10]==labels_pred[10*i:10*i+10]))